# Track A: Build a RAG Pipeline

**Retrieval-Augmented Generation** gives an LLM access to *your* data without retraining.

In this lab you will build a complete RAG pipeline from scratch:

1. **Chunk** a document into smaller pieces
2. **Embed** each chunk into a vector
3. **Store** the vectors in a FAISS index
4. **Retrieve** the most relevant chunks for a question
5. **Generate** an answer using Gemini with the retrieved context

**Stack:** `sentence-transformers` + `FAISS` + Google Gemini API (free)

**Time:** ~45 minutes

---

## 0. Setup

Install the required packages. This takes about 1 minute.

In [ ]:
!pip install -q sentence-transformers faiss-cpu google-genai

## 1. The Document

We need a knowledge base that the LLM does *not* already know about.

We'll use a fictional company FAQ -- **NovaTech Solutions** -- a made-up cloud platform.
Since this company doesn't exist, the LLM can't answer questions about it from its training data.
This makes it easy to see the difference RAG makes.

In [ ]:
DOCUMENT = """
NovaTech Solutions - Internal Knowledge Base
=============================================

About NovaTech
--------------
NovaTech Solutions is a cloud infrastructure company founded in 2019 in
Gothenburg, Sweden. We provide managed Kubernetes hosting, CI/CD pipelines,
and observability tools for mid-size European companies. Our platform is
called NovaCloud and currently serves over 340 customers across 12 countries.
The company was founded by Astrid Lindqvist and Erik Bergman, who previously
worked at Ericsson.

NovaCloud Platform
------------------
NovaCloud runs on bare-metal servers in three data centers: Gothenburg (primary),
Frankfurt, and Amsterdam. Each cluster runs Kubernetes 1.29 with Cilium for
networking and Longhorn for persistent storage. We do NOT use any public cloud
providers (AWS, Azure, GCP) -- all infrastructure is self-managed.

Customers connect via our CLI tool called `nova-cli` or through the web dashboard
at dashboard.novacloud.io. API access is available at api.novacloud.io/v2.

Pricing
-------
We offer three plans:
- Starter: EUR 49/month - 2 namespaces, 8 vCPU, 16 GB RAM, community support
- Professional: EUR 199/month - 10 namespaces, 32 vCPU, 64 GB RAM, email support with 4h SLA
- Enterprise: custom pricing - unlimited namespaces, dedicated nodes, 24/7 phone support, on-site SRE option

All plans include free ingress traffic. Egress is billed at EUR 0.05/GB after
the first 100 GB/month (free tier).

CI/CD Pipeline Service
----------------------
Our CI/CD service is called NovaPipe. It is built on top of Tekton and supports
GitHub, GitLab, and Bitbucket as source providers. Build environments run in
ephemeral containers with a maximum build time of 30 minutes on Starter plans
and 90 minutes on Professional/Enterprise plans.

NovaPipe configuration uses YAML files stored in the repository root as
`.novapipe.yml`. Each pipeline can have up to 20 steps. Artifacts are stored
in our built-in registry at registry.novacloud.io.

Observability Stack
-------------------
NovaWatch is our observability suite. It includes:
- Metrics: Prometheus-compatible, scraped every 15 seconds, retained for 90 days
- Logs: Loki-based log aggregation, retained for 30 days (90 days on Enterprise)
- Traces: OpenTelemetry-native distributed tracing, retained for 14 days
- Alerts: PagerDuty, Slack, email, and webhook integrations

Dashboards are available in the web UI and are compatible with Grafana JSON exports.

Authentication & Security
-------------------------
NovaCloud uses OIDC-based authentication. We support integration with:
- Azure AD / Entra ID
- Google Workspace
- Okta
- Any SAML 2.0 provider

All data at rest is encrypted with AES-256. Data in transit uses TLS 1.3.
We are SOC 2 Type II certified and GDPR compliant. Annual penetration tests
are conducted by an independent third party (currently: Truesec AB).

Support & SLAs
--------------
- Starter: community forum only (forum.novacloud.io)
- Professional: email support, 4-hour response SLA during business hours (CET)
- Enterprise: 24/7 phone + email, 30-minute critical response SLA, dedicated
  account manager, quarterly business reviews

Platform uptime SLA:
- Starter: 99.5%
- Professional: 99.9%
- Enterprise: 99.95% (with multi-region deployment)

Common Issues & Troubleshooting
-------------------------------
Q: My deployment is stuck in "Pending" state.
A: This usually means your namespace has hit its resource quota. Check your
   current usage with `nova-cli quota show` and request an increase via the
   dashboard or contact support.

Q: NovaPipe builds are failing with "OOMKilled".
A: Build containers have a 4 GB memory limit by default. You can increase this
   to 8 GB in your `.novapipe.yml` by adding `resources.memory: 8Gi` to the
   failing step. Enterprise customers can request up to 32 GB.

Q: How do I migrate from AWS EKS to NovaCloud?
A: We provide a migration tool called `nova-migrate` that exports your EKS
   workloads (deployments, services, configmaps, secrets) and re-applies them
   to a NovaCloud namespace. Run `nova-migrate scan --source eks` to get a
   compatibility report. Note: AWS-specific resources (ALB ingress, EBS volumes)
   will need manual adjustment.

Q: Can I bring my own domain name?
A: Yes. Add a CNAME record pointing to ingress.novacloud.io and configure the
   domain in your namespace settings. TLS certificates are automatically
   provisioned via Let's Encrypt.

Q: What container registries are supported?
A: Our built-in registry at registry.novacloud.io, Docker Hub, GitHub Container
   Registry (ghcr.io), and any OCI-compatible registry. Private registry
   credentials can be configured as namespace secrets.

Incident History (2024)
-----------------------
- Jan 15: Frankfurt DC network switch failure, 23-minute outage for Frankfurt customers
- Mar 02: NovaPipe build queue backlog due to registry storage full, 2-hour degraded service
- Jun 28: Gothenburg DC cooling failure, preventive failover to Frankfurt, no customer impact
- Sep 11: API rate limiter bug caused false 429 errors for 45 minutes
- Nov 30: Loki upgrade caused log ingestion delay of 6 hours for Enterprise customers
"""

print(f"Document length: {len(DOCUMENT)} characters")

## 2. Chunking

LLMs have limited context windows, and embedding models work best on shorter text.
We split the document into smaller **chunks**.

We'll use a simple strategy: split by double newlines (paragraph boundaries),
then merge small chunks together until they reach a target size.

**Key parameters:**
- `chunk_size` -- target number of characters per chunk
- `chunk_overlap` -- how many characters to repeat between chunks (helps preserve context at boundaries)

In [ ]:
def chunk_text(text, chunk_size=500, chunk_overlap=50):
    """
    Split text into overlapping chunks.

    Strategy: split on paragraph boundaries (double newlines),
    then merge paragraphs into chunks of roughly `chunk_size` characters.
    """
    # Split into paragraphs
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks = []
    current_chunk = ""

    for para in paragraphs:
        # If adding this paragraph would exceed chunk_size, save current chunk and start new one
        if current_chunk and len(current_chunk) + len(para) > chunk_size:
            chunks.append(current_chunk.strip())
            # Start new chunk with overlap from the end of the previous chunk
            overlap_text = current_chunk[-chunk_overlap:] if chunk_overlap > 0 else ""
            current_chunk = overlap_text + "\n\n" + para
        else:
            current_chunk = current_chunk + "\n\n" + para if current_chunk else para

    # Don't forget the last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

In [ ]:
chunks = chunk_text(DOCUMENT, chunk_size=500, chunk_overlap=50)

print(f"Created {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)
    print()

### Discussion: Chunking strategies

Our chunking is simple but it works. In production you might consider:

| Strategy | When to use |
|----------|-------------|
| Fixed-size with overlap | Simple, works for most text |
| Sentence-based | When paragraph boundaries aren't meaningful |
| Semantic chunking | Group sentences by meaning (expensive) |
| Document-structure aware | Markdown headers, HTML tags, etc. |

The `chunk_size` and `chunk_overlap` are hyperparameters. Try changing them later to see how it affects results!

## 3. Embedding

Now we convert each chunk into a **vector** (a list of numbers) that captures its meaning.

We'll use `sentence-transformers` with the `all-MiniLM-L6-v2` model:
- Small and fast (80 MB)
- 384-dimensional vectors
- Good quality for semantic search
- Runs fine on CPU

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Model loaded: {embedding_model}")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

In [ ]:
# Embed all chunks
chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True)

print(f"\nEmbedded {len(chunk_embeddings)} chunks")
print(f"Each embedding has {chunk_embeddings.shape[1]} dimensions")
print(f"\nFirst embedding (first 10 values): {chunk_embeddings[0][:10]}")

### What just happened?

Each chunk of text is now a vector of 384 numbers. These vectors live in a high-dimensional space where **similar meanings are close together**.

For example:
- "How much does it cost?" and "What is the pricing?" would have similar vectors
- "How much does it cost?" and "What is Kubernetes?" would have distant vectors

This is the core idea behind semantic search.

## 4. Vector Store (FAISS)

We need a way to quickly find the most similar vectors. That's what a **vector database** does.

[FAISS](https://github.com/facebookresearch/faiss) (Facebook AI Similarity Search) is a library for efficient similarity search. For our small dataset we'll use a simple flat index (brute-force search). In production with millions of vectors, you'd use approximate methods like IVF or HNSW.

In [ ]:
import faiss
import numpy as np

# Get the dimension of our embeddings
dimension = chunk_embeddings.shape[1]  # 384

# Create a FAISS index
index = faiss.IndexFlatL2(dimension)  # L2 = Euclidean distance

# Add our chunk embeddings to the index
index.add(chunk_embeddings.astype(np.float32))

print(f"FAISS index created with {index.ntotal} vectors of dimension {dimension}")

### Test retrieval

Let's test that retrieval works by searching for a question *before* we hook up the LLM.

In [ ]:
def retrieve(query, k=3):
    """
    Given a question, find the k most relevant chunks.

    Steps:
    1. Embed the query using the same model
    2. Search the FAISS index for nearest neighbors
    3. Return the matching chunks
    """
    # Embed the query
    query_embedding = embedding_model.encode([query]).astype(np.float32)

    # Search FAISS (returns distances and indices)
    distances, indices = index.search(query_embedding, k)

    # Collect results
    results = []
    for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        results.append({
            "chunk_id": int(idx),
            "distance": float(dist),
            "text": chunks[idx]
        })

    return results

In [ ]:
# Test: search for pricing information
query = "How much does NovaTech cost?"

results = retrieve(query, k=3)

print(f"Query: '{query}'\n")
for r in results:
    print(f"Chunk {r['chunk_id']} (distance: {r['distance']:.4f}):")
    print(r["text"][:200])
    print()

In [ ]:
# Test another query
query2 = "What happened when there were incidents?"

results2 = retrieve(query2, k=2)

print(f"Query: '{query2}'\n")
for r in results2:
    print(f"Chunk {r['chunk_id']} (distance: {r['distance']:.4f}):")
    print(r["text"][:200])
    print()

The retriever is finding relevant chunks. Notice how semantic search works -- we didn't search for exact keyword matches. The question "How much does NovaTech cost?" matches the pricing section even though the document doesn't use the word "cost".

---

## 5. Connect the LLM (Google Gemini)

Now we connect Google's Gemini API to generate answers based on the retrieved context.

### Get your API key

1. Go to [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
2. Click **Create API key**
3. Copy the key and paste it below

The free tier gives you 15 requests/minute with Gemini Flash -- more than enough for this lab.

In [ ]:
from google.colab import userdata

# Option 1 (recommended): Store your key in Colab Secrets
#   Click the key icon in the left sidebar -> Add "GEMINI_API_KEY"
#
# Option 2: Paste it directly (less secure, don't share the notebook after)
#   GEMINI_API_KEY = "paste-your-key-here"

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    print("API key loaded from Colab Secrets")
except Exception:
    GEMINI_API_KEY = input("Enter your Gemini API key: ")
    print("API key set manually")

In [ ]:
from google import genai

# Create the Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

# Quick test
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say 'hello' and nothing else."
)
print(f"Gemini says: {response.text}")

## 6. The RAG Pipeline

Now we put it all together. The key insight is in the **prompt construction**:

We take the retrieved chunks and inject them into the prompt as context,
then ask the LLM to answer based *only* on that context.

```
User question  -->  Embed  -->  Search FAISS  -->  Top-k chunks
                                                       |
                                                       v
                                              Build prompt with context
                                                       |
                                                       v
                                                 Gemini generates answer
```

In [ ]:
def rag_query(question, k=3, show_context=False):
    """
    Full RAG pipeline: retrieve relevant chunks and generate an answer.

    Args:
        question: The user's question
        k: Number of chunks to retrieve
        show_context: If True, also print the retrieved chunks
    """
    # Step 1: Retrieve relevant chunks
    results = retrieve(question, k=k)
    context = "\n\n---\n\n".join([r["text"] for r in results])

    if show_context:
        print("=" * 60)
        print("RETRIEVED CONTEXT:")
        print("=" * 60)
        for r in results:
            print(f"\n[Chunk {r['chunk_id']}, distance: {r['distance']:.4f}]")
            print(r["text"][:300])
        print("\n" + "=" * 60)

    # Step 2: Build the augmented prompt
    prompt = f"""You are a helpful assistant. Answer the question based ONLY on
the context provided below. If the context does not contain enough information
to answer, say "I don't have enough information to answer that."

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    # Step 3: Generate the answer
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text

### Try it out!

In [ ]:
answer = rag_query("What are the pricing plans?", show_context=True)
print("\nANSWER:")
print(answer)

In [ ]:
answer = rag_query("How do I migrate from AWS?", show_context=True)
print("\nANSWER:")
print(answer)

In [ ]:
answer = rag_query("My NovaPipe build is running out of memory. What should I do?")
print(answer)

## 7. Without RAG vs With RAG

Let's compare what happens when we ask Gemini about NovaTech *without* the retrieved context.
Since NovaTech is fictional, the LLM should either hallucinate or admit it doesn't know.

In [ ]:
def ask_without_rag(question):
    """Ask the LLM directly, no retrieved context."""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=question,
    )
    return response.text

In [ ]:
test_question = "What is the uptime SLA for NovaTech's Professional plan?"

print("=" * 60)
print("WITHOUT RAG (LLM only):")
print("=" * 60)
print(ask_without_rag(test_question))

print("\n")

print("=" * 60)
print("WITH RAG (retrieved context + LLM):")
print("=" * 60)
print(rag_query(test_question))

In [ ]:
test_question2 = "Who founded NovaTech and where are their data centers?"

print("=" * 60)
print("WITHOUT RAG:")
print("=" * 60)
print(ask_without_rag(test_question2))

print("\n")

print("=" * 60)
print("WITH RAG:")
print("=" * 60)
print(rag_query(test_question2))

### What did we observe?

- **Without RAG:** The LLM either hallucinated (made up plausible-sounding but wrong information) or said it didn't know
- **With RAG:** The LLM gave accurate answers grounded in the actual document

This is the core value of RAG -- the model answers from *your* data, not from its training data.

---

## 8. Explore & Experiment

You've built a complete RAG pipeline! Here are things to try:

### Try your own questions

In [ ]:
# Try your own questions about NovaTech!
your_question = "What security certifications does NovaTech have?"

print(rag_query(your_question, show_context=True))

### Experiment: Change the number of retrieved chunks

What happens if you retrieve more or fewer chunks? Try `k=1` vs `k=5`.

In [ ]:
question = "Tell me about NovaTech's observability features and pricing."

print("--- k=1 (only 1 chunk) ---")
print(rag_query(question, k=1))

print("\n--- k=5 (5 chunks) ---")
print(rag_query(question, k=5))

### Experiment: Change the chunking parameters

Try re-running the pipeline with different chunk sizes. You'll need to re-embed and re-index.

In [ ]:
# Try smaller chunks (more precise retrieval, less context per chunk)
small_chunks = chunk_text(DOCUMENT, chunk_size=200, chunk_overlap=30)
print(f"Small chunks: {len(small_chunks)} chunks")

# Try larger chunks (more context per chunk, less precise retrieval)
large_chunks = chunk_text(DOCUMENT, chunk_size=1000, chunk_overlap=100)
print(f"Large chunks: {len(large_chunks)} chunks")

print(f"\nOriginal: {len(chunks)} chunks (500 char target)")

### Experiment: Test the boundaries

Try asking something that is NOT in the document. Does the model correctly say it doesn't know?

In [ ]:
# This information is NOT in the document
print(rag_query("What programming languages does NovaTech's API support?"))
print()
print(rag_query("How many employees does NovaTech have?"))

### Bonus: Visualize the embeddings

We can use dimensionality reduction to see our chunks in 2D space.
Chunks that are close together have similar meaning.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reduce 384 dimensions to 2
pca = PCA(n_components=2)
reduced = pca.fit_transform(chunk_embeddings)

plt.figure(figsize=(10, 7))
plt.scatter(reduced[:, 0], reduced[:, 1], s=100, c='steelblue', edgecolors='white', linewidth=1.5)

for i in range(len(chunks)):
    # Show first few words as label
    label = chunks[i][:40].replace('\n', ' ') + "..."
    plt.annotate(label, (reduced[i, 0], reduced[i, 1]),
                 fontsize=8, ha='center', va='bottom')

plt.title("Document chunks in 2D (PCA projection)")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.tight_layout()
plt.show()

---

## Recap

You built a RAG pipeline from scratch:

| Step | What we did | Tool |
|------|------------|------|
| Chunk | Split the document into pieces | Python |
| Embed | Convert chunks to vectors | sentence-transformers |
| Store | Index vectors for fast search | FAISS |
| Retrieve | Find relevant chunks for a query | FAISS |
| Generate | Answer using retrieved context | Gemini API |

### In production, you would also consider:

- **Better chunking:** document-structure-aware splitting, recursive text splitters
- **Hybrid search:** combine vector search with keyword search (BM25)
- **Re-ranking:** use a cross-encoder to re-score retrieved chunks
- **Managed vector DBs:** Pinecone, Weaviate, Qdrant, pgvector
- **Evaluation:** measuring retrieval quality (precision@k, recall@k)
- **Metadata filtering:** filter by date, source, category before vector search

### Key takeaway

RAG lets you give an LLM access to your private data **without retraining**.
It's the most practical and widely-used customization technique today.